
# Carpeta 1 — Extracción de Reglas con el Algoritmo PRISM

**Proyecto:** Optimización Ambiental de la Semaforización Urbana mediante un Sistema de
Inferencia Difusa Multivariable (MIMO), Algoritmos Genéticos y Extracción de Reglas con PRISM.

**Universidad Técnica de Ambato — Facultad de Ingeniería en Sistemas, Electrónica e Industrial**
**Carrera de Software — Inteligencia Artificial**

**Autores:** Guevara Panimboza Marlon Steven, Fiallos Yanza Jonathan Josué

---

## Planteamiento del problema

Los semáforos de la mayoría de intersecciones urbanas operan con **tiempos fijos**,
predefinidos sin considerar las condiciones reales de tráfico en cada momento del día. Esto
provoca dos problemas concretos y medibles en el dataset `smart_city_traffic_mobility.csv`:

* **Ineficiencia operativa:** en horas de congestión los ciclos fijos generan colas
 (`queue_length`) y tiempos de espera excesivos, mientras que en horas de tráfico fluido el
 mismo tiempo de luz verde se desperdicia, reduciendo la capacidad global de la vía.
* **Impacto ambiental:** los vehículos detenidos o circulando a baja velocidad
 (`average_speed` baja) incrementan las emisiones (`emission_estimate`) y deterioran la
 calidad del aire (`air_quality_index`) de la zona.

Ajustar el tiempo de luz verde y el ciclo del semáforo (`green_light_duration`,
`signal_cycle_adjustment`) en función de las condiciones de tráfico observadas
(`traffic_density`, `queue_length`, `vehicle_count`, `average_speed`) es un problema de
**control con incertidumbre y conocimiento experto expresable en reglas lingüísticas** ("si
hay mucha densidad Y la cola es larga, entonces extender el verde"), lo que lo hace apto para
un **sistema de inferencia difusa**. Sin embargo, definir esas reglas y los umbrales de las
categorías lingüísticas (baja/media/alta, corta/moderada/extensa, etc.) manualmente es
subjetivo y difícil de justificar con un dataset de 204 000 registros reales.



## Objetivo de esta libreta

Esta libreta implementa el **algoritmo PRISM** a partir del dataset real
`smart_city_traffic_mobility.csv` (204 000 registros), las **reglas de clasificación** que
relacionan las 4 variables de entrada del tráfico con:

1. `green_light_duration` (discretizada en **Corta / Media / Larga**)
2. `signal_cycle_adjustment` (variable derivada de `signal_cycle_seconds`, discretizada en
 **Reducir / Mantener / Extender** )

El resultado de esta libreta (las reglas PRISM en formato `.json`/`.csv`) es el **insumo
directo** que utilizará la Carpeta 2 (`02_Sistema_Difuso_MIMO`) para construir la base de
reglas del sistema difuso Mamdani MIMO

---

## Marco teórico 

 En este proyecto se
usan **reglas de clasificación** (el consecuente es siempre `green_light_duration` o
`signal_cycle_adjustment`).

**Medidas de evaluación de reglas** (idénticas a las vistas en clase):

| Medida | Fórmula | Significado |
|---|---|---|
| **Confianza** | $\text{Confianza}(A\to B) = P(B\mid A) = \dfrac{N(A \wedge B)}{N(A)}$ | Probabilidad de que se cumpla el consecuente dado el antecedente |
| **Soporte** | $\text{Soporte}(A\to B) = P(A \cap B) = \dfrac{N(A \wedge B)}{N}$ | Frecuencia relativa de la regla sobre el total de datos |
| **Cobertura** | $N(A \wedge B)$ | Número absoluto de ejemplos que cumplen antecedente y consecuente |
| **Lift** | $\text{lift}(A,B) = \dfrac{\text{Confianza}(A\to B)}{P(B)}$ | Correlación entre A y B: lift > 1 indica que A hace más probable B |

**Algoritmo PRISM — recubrimiento secuencial basado en **confianza**, con
desempate por **cobertura**.

La función `prism()` implementada en el **Paso 5** de esta libreta es una traducción directa
de este pseudocódigo a Python (vectorizado con `pandas` para manejar 204 000 registros)

## Paso 1 — Importación de librerías

In [2]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/smart_city_traffic_mobility.csv') if Path('../data/smart_city_traffic_mobility.csv').exists() else Path('data/smart_city_traffic_mobility.csv')
assert DATA_PATH.exists(), f"No se encontró el archivo en {DATA_PATH.resolve()}"
df = pd.read_csv(DATA_PATH, sep=';')
print(f"Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas.")


Dataset cargado correctamente: 204000 filas, 47 columnas.



## Paso 2 — Carga del dataset

El archivo `smart_city_traffic_mobility.csv` se encuentra en la carpeta compartida `../data/`
(un nivel arriba de esta libreta) para evitar duplicar 48 MB de datos en cada una de las tres
carpetas del proyecto. El separador del archivo es `;`.


In [3]:

DATA_PATH = Path('../data/smart_city_traffic_mobility.csv') if Path('../data/smart_city_traffic_mobility.csv').exists() else Path('data/smart_city_traffic_mobility.csv')
assert DATA_PATH.exists(), f"No se encontró el archivo en {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH, sep=';')
print(f"Registros: {len(df):,}")
print(f"Columnas : {df.shape[1]}")
df.head()


Registros: 204,000
Columnas : 47


,record_id,city_zone,road_id,intersection_id,latitude,longitude,road_type,lanes,speed_limit,vehicle_count,...,traffic_camera_count,iot_sensor_health,signal_cycle_seconds,green_light_duration,emergency_vehicle_detected,traffic_flow_rate,congestion_score,emission_estimate,fuel_waste_estimate,congestion_level
0,REC_0000001,Financial District,RD_206,INT_001,40.757.551,-73.990.504,Local Street,2,40,162,...,1,Optimal,60,18,0,162.0,16.8,39.14,8.53,Low
1,REC_0000002,Financial District,RD_206,INT_001,40.757.551,-73.990.504,Local Street,2,40,150,...,1,Optimal,60,18,0,150.0,18.4,36.09,7.83,Low
2,REC_0000003,Financial District,RD_206,INT_001,40.757.551,-73.990.504,Local Street,2,40,186,...,1,Optimal,60,18,0,186.0,20.7,44.01,9.39,Low
3,REC_0000004,Financial District,RD_206,INT_001,40.757.551,-73.990.504,Local Street,2,40,162,...,1,Optimal,60,18,0,162.0,17.8,38.47,8.24,Low
4,REC_0000005,Financial District,RD_206,INT_001,40.757.551,-73.990.504,Local Street,2,40,168,...,1,Optimal,60,18,0,168.0,17.3,40.28,8.71,Low


In [4]:

# Columnas relevantes para este proyecto (definidas en la Sección V del documento)
COLS_ENTRADA = ['traffic_density', 'queue_length', 'vehicle_count', 'average_speed']
COLS_SALIDA_NUM = ['green_light_duration', 'signal_cycle_seconds']
COLS_AMBIENTALES = ['emission_estimate', 'air_quality_index']

df[COLS_ENTRADA + COLS_SALIDA_NUM + COLS_AMBIENTALES].describe().T


,count,mean,std,min,25%,50%,75%,max
traffic_density,204000.0,90.660393,85.277658,9.00,20.80,35.70,175.10,378.50
queue_length,204000.0,121.470659,230.578569,0.00,6.80,19.10,173.80,10111.60
vehicle_count,204000.0,749.585907,754.542368,54.00,171.00,308.00,1208.00,3162.00
average_speed,204000.0,42.235582,22.214697,5.00,27.70,40.00,58.10,90.00
green_light_duration,204000.0,39.070775,27.153351,18.00,18.00,23.00,84.00,84.00
signal_cycle_seconds,204000.0,75.174118,26.080578,60.00,60.00,60.00,120.00,120.00
emission_estimate,204000.0,257.416941,339.096417,12.74,41.25,74.07,376.84,9717.22
air_quality_index,204000.0,72.517485,20.343502,20.00,58.00,72.00,87.00,152.00



## Paso 3 — Discretización de las variables de entrada

PRISM trabaja únicamente con **atributos categóricos**, por lo que cada variable numérica de
entrada se discretiza en **3 categorías de tamaño aproximadamente similar**, usando los
percentiles **33.3 %** y **66.7 %** del propio dataset como puntos de corte. Estos mismos cortes servirán después como referencia para ubicar
los vértices de las funciones de pertenencia difusas en la Carpeta 2.


In [5]:
ETIQUETAS_ENTRADA = {
    'traffic_density': ['Baja', 'Moderada', 'Alta'],
    'queue_length': ['Corta', 'Moderada', 'Extensa'],
    'vehicle_count': ['Bajo', 'Medio', 'Alto'],
    'average_speed': ['Baja', 'Media', 'Alta'],
}

def discretizar_terciles(serie: pd.Series, etiquetas: list[str]):
    u1, u2 = serie.quantile([1/3, 2/3])
    def etiquetar(x):
        if x <= u1:
            return etiquetas[0]
        elif x <= u2:
            return etiquetas[1]
        else:
            return etiquetas[2]
    return serie.apply(etiquetar), (float(u1), float(u2))

umbrales = {}
for col, etiquetas in ETIQUETAS_ENTRADA.items():
    df[col + '_lbl'], umbrales[col] = discretizar_terciles(df[col], etiquetas)

print("Discretización de variables de entrada completada correctamente.")


Discretización de variables de entrada completada correctamente.


In [6]:
# Distribución de frecuencias para las etiquetas de entrada
for col in ETIQUETAS_ENTRADA.keys():
    print(f"\n--- {col} --- ")
    print(df[col + '_lbl'].value_counts(normalize=True).round(4) * 100)



--- traffic_density --- 
traffic_density_lbl
Baja        33.44
Alta        33.33
Moderada    33.22
Name: proportion, dtype: float64

--- queue_length --- 
queue_length_lbl
Corta       33.43
Extensa     33.32
Moderada    33.25
Name: proportion, dtype: float64

--- vehicle_count --- 
vehicle_count_lbl
Bajo     33.43
Alto     33.32
Medio    33.25
Name: proportion, dtype: float64

--- average_speed --- 
average_speed_lbl
Baja     33.36
Media    33.34
Alta     33.30
Name: proportion, dtype: float64



## Paso 4 — Discretización de las variables de salida

### 4.1 `green_light_duration`

Se aplica el mismo criterio de terciles :
**Corta ≤ 18 s**, **19 s ≤ Media ≤ 40 s**, **Larga > 40 s**.


In [7]:
# Discretizar variable de salida green_light_duration
df['green_light_duration_lbl'], gld_umbral = discretizar_terciles(df['green_light_duration'], ['Corta', 'Media', 'Larga'])
umbrales['green_light_duration'] = gld_umbral

print(f"Umbrales (terciles) para green_light_duration: {gld_umbral}")
print(df['green_light_duration_lbl'].value_counts())
print(df['green_light_duration_lbl'].value_counts(normalize=True).round(4) * 100)


Umbrales (terciles) para green_light_duration: (18.0, 40.0)
green_light_duration_lbl
Corta    75251
Larga    64625
Media    64124
Name: count, dtype: int64
green_light_duration_lbl
Corta    36.89
Larga    31.68
Media    31.43
Name: proportion, dtype: float64



### 4.2 `signal_cycle_adjustment` (variable derivada)

El dataset **no contiene** una columna `signal_cycle_adjustment` con las etiquetas
Reducir/Mantener/Extender; solo existe `signal_cycle_seconds`, que toma únicamente dos valores
(**60 y 120 segundos**).

**Criterio de ingeniería aplicado:** cada intersección (`intersection_id`) tiene un ciclo
"típico" (su propio promedio histórico de `signal_cycle_seconds`, calculado sobre todos sus
registros). Se compara el ciclo del registro actual contra ese promedio propio:

* Si el ciclo actual es **claramente mayor** que su promedio histórico (Δ > 15 s) → `Extender`
* Si el ciclo actual es **claramente menor** que su promedio histórico (Δ < −15 s) → `Reducir`
* En caso contrario (cerca de su propio promedio) → `Mantener`

Este criterio es razonable porque conserva la semántica de la variable original (una
*decisión relativa* de ajuste de ciclo) sin inventar información que no está en los datos: la
decisión de "Extender/Reducir/Mantener" se basa únicamente en el propio histórico de cada
intersección.


In [8]:
ciclo_promedio_interseccion = df.groupby('intersection_id')['signal_cycle_seconds'].transform('mean')
delta_ciclo = df['signal_cycle_seconds'] - ciclo_promedio_interseccion

UMBRAL_DELTA = 15.0

def etiquetar_ajuste(delta):
    if delta > UMBRAL_DELTA:
        return 'Extender'
    elif delta < -UMBRAL_DELTA:
        return 'Reducir'
    else:
        return 'Mantener'

df['signal_cycle_adjustment_lbl'] = delta_ciclo.apply(etiquetar_ajuste)
print(df['signal_cycle_adjustment_lbl'].value_counts())
print(df['signal_cycle_adjustment_lbl'].value_counts(normalize=True).round(4) * 100)


signal_cycle_adjustment_lbl
Reducir     83838
Mantener    68570
Extender    51592
Name: count, dtype: int64
signal_cycle_adjustment_lbl
Reducir     41.10
Mantener    33.61
Extender    25.29
Name: proportion, dtype: float64



## Paso 5 — Implementación del algoritmo PRISM

PRISM (Cendrowska, 1987) es un algoritmo de **recubrimiento secuencial** ("separate-and-conquer")
que construye reglas `SI (antecedente) ENTONCES clase = C` **directamente** de los datos, sin
pasar por un árbol de decisión. Para una clase objetivo `C` fija, el algoritmo es:

Se implementa a continuación de forma genérica, calculando además **Cobertura** (`p`),
**Soporte** (`p / N`) y **Lift** (`confianza / P(C)`). La función admite un modo `verbose=True` que imprime, iteración por
iteración.



## Paso 6 — Ejecución detallada (modo `verbose`) para la clase

Se ejecuta primero, **con todo el detalle iteración por iteración**, la extracción de reglas
para la clase `green_light_duration = Larga`,(Iteración 1: `vehicle_count = Alto`; Iteración 2:
`average_speed = Alta`; Iteración 3: `traffic_density = Alta` ⇒ confianza 1.00).


In [9]:
from dataclasses import dataclass, field

@dataclass
class ReglaPRISM:
    antecedente: list # lista de tuplas (atributo, valor)
    clase_objetivo: str
    variable_objetivo: str
    confianza: float
    cobertura: int # p
    soporte: float # p / N
    lift: float
    t: int # N(A) total de instancias cubiertas por el antecedente

    def to_dict(self):
        return {
            'antecedente': self.antecedente,
            'clase_objetivo': self.clase_objetivo,
            'variable_objetivo': self.variable_objetivo,
            'confianza': round(self.confianza, 6),
            'cobertura': self.cobertura,
            'soporte': round(self.soporte, 6),
            'lift': round(self.lift, 4),
            't': self.t,
            'regla_str': self.__repr__()
        }

def prism(df: pd.DataFrame, atributos: dict[str, list[str]], columna_objetivo: str, clase_objetivo: str,
          cobertura_minima: int = 200, verbose: bool = False) -> list[ReglaPRISM]:
    N = len(df)
    P_prior = (df[columna_objetivo] == clase_objetivo).mean()
    E = df.copy()
    reglas: list[ReglaPRISM] = []
    num_regla = 0

    while (E[columna_objetivo] == clase_objetivo).sum() > cobertura_minima:
        num_regla += 1
        subconjunto = E
        atributos_usados = set()
        antecedente = []
        num_iter = 0

        if verbose:
            print(f"\n{'='*70}\nRegla #{num_regla} — instancias de '{clase_objetivo}' "
                  f"aún sin cubrir: {(E[columna_objetivo] == clase_objetivo).sum():,}\n{'='*70}")

        while True:
            num_iter += 1
            candidatos = []
            for attr, valores in atributos.items():
                if attr in atributos_usados:
                    continue
                for v in valores:
                    mask = subconjunto[attr] == v
                    t = int(mask.sum())
                    if t == 0:
                        continue
                    p = int((subconjunto.loc[mask, columna_objetivo] == clase_objetivo).sum())
                    conf = p / t
                    lift = (conf / P_prior) if P_prior > 0 else 0.0
                    candidatos.append({
                        'condicion': f"{attr} = {v}", 'attr': attr, 'valor': v,
                        'N(A)': t, 'N(A^clase)': p, 'confianza': conf,
                        'cobertura': p, 'soporte': p / N, 'lift': lift,
                    })

            if not candidatos:
                break

            tabla_cand = pd.DataFrame(candidatos).sort_values(
                ['confianza', 'cobertura'], ascending=False
            ).reset_index(drop=True)

            if verbose:
                print(f"\n-- Iteración {num_iter} (antecedente parcial: "
                      f"{antecedente if antecedente else '∅'}) --")
                print(tabla_cand[['condicion', 'N(A)', 'N(A^clase)', 'confianza',
                                  'cobertura', 'soporte', 'lift']].round(4).to_string(index=False))

            mejor = tabla_cand.iloc[0]
            antecedente.append((mejor['attr'], mejor['valor']))
            atributos_usados.add(mejor['attr'])
            subconjunto = subconjunto[subconjunto[mejor['attr']] == mejor['valor']]

            if mejor['confianza'] >= 0.999999 or len(atributos_usados) == len(atributos):
                if verbose:
                    motivo = 'confianza perfecta (1.00)' if mejor['confianza'] >= 0.999999                         else 'se usaron todos los atributos disponibles'
                    print(f"\n>> Se detiene la especialización de esta regla: {motivo}.")
                break

        if not antecedente:
            break

        mask_final = pd.Series(True, index=E.index)
        for attr, v in antecedente:
            mask_final &= (E[attr] == v)
        t_final = int(mask_final.sum())
        p_final = int((E.loc[mask_final, columna_objetivo] == clase_objetivo).sum())
        conf_final = p_final / t_final if t_final else 0.0
        lift_final = (conf_final / P_prior) if P_prior > 0 else 0.0

        regla = ReglaPRISM(
            antecedente=antecedente, clase_objetivo=clase_objetivo,
            variable_objetivo=columna_objetivo.replace('_lbl', ''),
            confianza=conf_final, cobertura=p_final, soporte=p_final / N,
            lift=lift_final, t=t_final,
        )
        reglas.append(regla)
        if verbose:
            print(f"\n>>> REGLA FINAL #{num_regla}: {regla}")

        E = E[~mask_final]
        if len(E) == 0:
            break

    return reglas

print('Función prism() definida correctamente.')


Función prism() definida correctamente.


In [10]:

ATRIBUTOS_ENTRADA = {
 'traffic_density_lbl': ['Baja', 'Moderada', 'Alta'],
 'queue_length_lbl': ['Corta', 'Moderada', 'Extensa'],
 'vehicle_count_lbl': ['Bajo', 'Medio', 'Alto'],
 'average_speed_lbl': ['Baja', 'Media', 'Alta'],
}

COBERTURA_MINIMA = max(150, int(0.001 * len(df))) # ~0.1% de N, evita reglas triviales
print('Cobertura mínima de parada:', COBERTURA_MINIMA)

reglas_larga = prism(df, ATRIBUTOS_ENTRADA, 'green_light_duration_lbl', 'Larga',
 cobertura_minima=COBERTURA_MINIMA, verbose=True)


Cobertura mínima de parada: 204



Regla #1 — instancias de 'Larga' aún sin cubrir: 64,625

-- Iteración 1 (antecedente parcial: ∅) --
                     condicion  N(A)  N(A^clase)  confianza  cobertura  soporte   lift
      vehicle_count_lbl = Alto 67970       64625     0.9508      64625   0.3168 3.0013
    traffic_density_lbl = Alta 68000       60943     0.8962      60943   0.2987 2.8291
    queue_length_lbl = Extensa 67971       56951     0.8379      56951   0.2792 2.6449
      average_speed_lbl = Baja 68053       50479     0.7418      50479   0.2474 2.3415
     average_speed_lbl = Media 68005        9483     0.1394       9483   0.0465 0.4402
      average_speed_lbl = Alta 67942        4663     0.0686       4663   0.0229 0.2166
   queue_length_lbl = Moderada 67829        4431     0.0653       4431   0.0217 0.2062
traffic_density_lbl = Moderada 67778        3682     0.0543       3682   0.0180 0.1715
      queue_length_lbl = Corta 68200        3243     0.0476       3243   0.0159 0.1501
    traffic_density_lbl = Baj

In [11]:

print(f"\n\nTotal de reglas obtenidas para 'Larga': {len(reglas_larga)}")
for r in reglas_larga:
 print(' -', r)




Total de reglas obtenidas para 'Larga': 13
 - ReglaPRISM(antecedente=[('vehicle_count_lbl', 'Alto'), ('average_speed_lbl', 'Alta'), ('traffic_density_lbl', 'Alta')], clase_objetivo='Larga', variable_objetivo='green_light_duration', confianza=1.0, cobertura=1679, soporte=0.008230392156862745, lift=np.float64(3.156673114119923), t=1679)
 - ReglaPRISM(antecedente=[('vehicle_count_lbl', 'Alto'), ('average_speed_lbl', 'Alta'), ('queue_length_lbl', 'Moderada'), ('traffic_density_lbl', 'Moderada')], clase_objetivo='Larga', variable_objetivo='green_light_duration', confianza=0.9900142653352354, cobertura=1388, soporte=0.006803921568627451, lift=np.float64(3.125151413978925), t=1402)
 - ReglaPRISM(antecedente=[('vehicle_count_lbl', 'Alto'), ('average_speed_lbl', 'Alta'), ('queue_length_lbl', 'Corta'), ('traffic_density_lbl', 'Moderada')], clase_objetivo='Larga', variable_objetivo='green_light_duration', confianza=0.9865913902611151, cobertura=1398, soporte=0.0068529411764705886, lift=np.float


## Paso 7 — Recubrimiento completo para las 3 clases de `green_light_duration`

Ahora se ejecuta PRISM (en modo silencioso) para las **tres** clases de la variable objetivo
`green_light_duration_lbl`, completando así la base total de reglas.


In [12]:

reglas_gld = {}
for clase in ['Corta', 'Media', 'Larga']:
 print(f"Extrayendo reglas para green_light_duration = {clase} ...")
 reglas_gld[clase] = prism(df, ATRIBUTOS_ENTRADA, 'green_light_duration_lbl', clase,
 cobertura_minima=COBERTURA_MINIMA, verbose=False)
 print(f" -> {len(reglas_gld[clase])} reglas obtenidas\n")

todas_reglas_gld = [r for lista in reglas_gld.values() for r in lista]
print(f"TOTAL reglas green_light_duration: {len(todas_reglas_gld)}")


Extrayendo reglas para green_light_duration = Corta ...
 -> 13 reglas obtenidas

Extrayendo reglas para green_light_duration = Media ...
 -> 20 reglas obtenidas

Extrayendo reglas para green_light_duration = Larga ...
 -> 13 reglas obtenidas

TOTAL reglas green_light_duration: 46


In [13]:

resumen_gld = pd.DataFrame([r.to_dict() for r in todas_reglas_gld])
resumen_gld['antecedente_txt'] = resumen_gld['antecedente'].apply(
 lambda ant: ' Y '.join([f"{a.replace('_lbl','')}={v}" for a, v in ant])
)
resumen_gld = resumen_gld.sort_values(['clase_objetivo', 'confianza'], ascending=[True, False])
resumen_gld[['clase_objetivo', 'antecedente_txt', 'confianza', 'cobertura', 'soporte', 'lift']]


,clase_objetivo,antecedente_txt,confianza,cobertura,soporte,lift
0,Corta,vehicle_count=Bajo,1.000000,68204,0.334333,2.7109
1,Corta,traffic_density=Baja Y average_speed=Media,1.000000,157,0.000770,2.7109
2,Corta,traffic_density=Baja Y average_speed=Baja Y qu...,0.333333,1,0.000005,0.9036
3,Corta,traffic_density=Baja Y queue_length=Extensa Y ...,0.268877,146,0.000716,0.7289
4,Corta,traffic_density=Baja Y queue_length=Moderada Y...,0.268222,1299,0.006368,0.7271
5,Corta,traffic_density=Baja Y queue_length=Corta Y ve...,0.246697,1251,0.006132,0.6688
6,Corta,average_speed=Media Y vehicle_count=Medio Y qu...,0.131439,1041,0.005103,0.3563
7,Corta,traffic_density=Moderada Y average_speed=Media...,0.126293,977,0.004789,0.3424
11,Corta,traffic_density=Moderada Y queue_length=Extens...,0.119907,103,0.000505,0.3251
8,Corta,average_speed=Alta Y vehicle_count=Medio Y que...,0.069307,105,0.000515,0.1879



## Paso 8 — Recubrimiento completo para las 3 clases de `signal_cycle_adjustment`

Se repite exactamente el mismo procedimiento para la variable derivada
`signal_cycle_adjustment_lbl` (Reducir / Mantener / Extender).


In [14]:

reglas_sca = {}
for clase in ['Reducir', 'Mantener', 'Extender']:
 print(f"Extrayendo reglas para signal_cycle_adjustment = {clase} ...")
 reglas_sca[clase] = prism(df, ATRIBUTOS_ENTRADA, 'signal_cycle_adjustment_lbl', clase,
 cobertura_minima=COBERTURA_MINIMA, verbose=False)
 print(f" -> {len(reglas_sca[clase])} reglas obtenidas\n")

todas_reglas_sca = [r for lista in reglas_sca.values() for r in lista]
print(f"TOTAL reglas signal_cycle_adjustment: {len(todas_reglas_sca)}")


Extrayendo reglas para signal_cycle_adjustment = Reducir ...
 -> 39 reglas obtenidas

Extrayendo reglas para signal_cycle_adjustment = Mantener ...
 -> 25 reglas obtenidas

Extrayendo reglas para signal_cycle_adjustment = Extender ...
 -> 12 reglas obtenidas

TOTAL reglas signal_cycle_adjustment: 76


In [15]:

resumen_sca = pd.DataFrame([r.to_dict() for r in todas_reglas_sca])
resumen_sca['antecedente_txt'] = resumen_sca['antecedente'].apply(
 lambda ant: ' Y '.join([f"{a.replace('_lbl','')}={v}" for a, v in ant])
)
resumen_sca = resumen_sca.sort_values(['clase_objetivo', 'confianza'], ascending=[True, False])
resumen_sca[['clase_objetivo', 'antecedente_txt', 'confianza', 'cobertura', 'soporte', 'lift']]


,clase_objetivo,antecedente_txt,confianza,cobertura,soporte,lift
64,Extender,vehicle_count=Alto Y traffic_density=Alta Y av...,1.000000,526,0.002578,3.9541
65,Extender,vehicle_count=Alto Y traffic_density=Alta Y av...,1.000000,243,0.001191,3.9541
70,Extender,vehicle_count=Alto Y traffic_density=Alta Y av...,1.000000,12,0.000059,3.9541
66,Extender,vehicle_count=Alto Y traffic_density=Alta Y av...,0.998901,909,0.004456,3.9498
67,Extender,vehicle_count=Alto Y traffic_density=Alta Y av...,0.991315,5935,0.029093,3.9198
...,...,...,...,...,...,...
33,Reducir,vehicle_count=Medio Y queue_length=Moderada Y ...,0.049101,112,0.000549,0.1195
34,Reducir,traffic_density=Baja Y queue_length=Corta Y ve...,0.042940,52,0.000255,0.1045
35,Reducir,vehicle_count=Medio Y average_speed=Baja Y tra...,0.036669,48,0.000235,0.0892
36,Reducir,vehicle_count=Medio Y queue_length=Corta Y tra...,0.022222,1,0.000005,0.0541


## Paso 9 — Filtrado de calidad de reglas

### Cálculo de la confianza

Para cada regla, la confianza se calcula como la proporción de ejemplos que cumplen el consecuente entre todos los ejemplos que cumplen el antecedente:

$$\text{Confianza}(A \rightarrow B) = \frac{N(A \cap B)}{N(A)} = \frac{p}{t}$$

En esta libreta, `p` corresponde a `cobertura` (ejemplos correctamente clasificados) y `t` corresponde al total de ejemplos cubiertos por el antecedente.

Antes de exportar la base de reglas al sistema difuso (Carpeta 2), se marca cada regla como **de alta confianza** (`confianza ≥ 0.60`) o **de respaldo** (`confianza < 0.60`). El valor 0.60 se utiliza como un criterio intermedio: exige que al menos 6 de cada 10 ejemplos cubiertos por una regla pertenezcan a la clase predicha, pero evita eliminar reglas potencialmente útiles debido a la variabilidad natural del tráfico urbano. Solo las reglas de alta confianza se usarán como reglas difusas principales; las de respaldo quedan documentadas mas no se emplean en el motor de inferencia, para evitar introducir ruido.

Este umbral es un criterio heurístico del proyecto y puede validarse comparándolo con otros valores, como 0.50, 0.70 y 0.80, mediante el desempeño final del sistema difuso.

In [18]:
UMBRAL_CONFIANZA_USO = 0.60

# Verificación explícita: confianza = cobertura / total de ejemplos cubiertos
resumen_gld['confianza_calculada'] = resumen_gld['cobertura'] / resumen_gld['t']
resumen_sca['confianza_calculada'] = resumen_sca['cobertura'] / resumen_sca['t']

resumen_gld['usar_en_difuso'] = resumen_gld['confianza_calculada'] >= UMBRAL_CONFIANZA_USO
resumen_sca['usar_en_difuso'] = resumen_sca['confianza_calculada'] >= UMBRAL_CONFIANZA_USO

print('green_light_duration -> reglas utilizables:', resumen_gld['usar_en_difuso'].sum(),
      '/', len(resumen_gld))
print('signal_cycle_adjustment -> reglas utilizables:', resumen_sca['usar_en_difuso'].sum(),
      '/', len(resumen_sca))

green_light_duration -> reglas utilizables: 28 / 46
signal_cycle_adjustment -> reglas utilizables: 39 / 76



## Paso 10 — Exportación de la base de reglas

Se guardan los resultados en tres archivos dentro de esta misma carpeta, para que la Carpeta 2
(`02_Sistema_Difuso_MIMO`) los consuma directamente:

* `reglas_prism_green_light_duration.json`
* `reglas_prism_signal_cycle_adjustment.json`
* `umbrales_discretizacion.json` (los cortes de las variables de entrada, reutilizados como
 referencia para ubicar las funciones de pertenencia difusas)
* `resumen_reglas_prism.csv` (tabla consolidada, legible en Excel)


In [17]:
OUT_DIR = Path('.')

with open(OUT_DIR / 'reglas_prism_green_light_duration.json', 'w', encoding='utf-8') as f:
    json.dump([r.to_dict() for r in todas_reglas_gld], f, ensure_ascii=False, indent=2)

with open(OUT_DIR / 'reglas_prism_signal_cycle_adjustment.json', 'w', encoding='utf-8') as f:
    json.dump([r.to_dict() for r in todas_reglas_sca], f, ensure_ascii=False, indent=2)

umbrales_entradas = {k: v for k, v in umbrales.items() if k != 'green_light_duration'}

with open(OUT_DIR / 'umbrales_discretizacion.json', 'w', encoding='utf-8') as f:
    json.dump({
        'entradas': umbrales_entradas,
        'green_light_duration': {
            'corte_1': gld_umbral[0],
            'corte_2': gld_umbral[1],
            'rango_min': float(df['green_light_duration'].min()),
            'rango_max': float(df['green_light_duration'].max())
        },
        'signal_cycle_adjustment': {
            'umbral_delta': UMBRAL_DELTA,
            'rango_delta_min': float(delta_ciclo.min()),
            'rango_delta_max': float(delta_ciclo.max())
        },
    }, f, ensure_ascii=False, indent=2)

tabla_final = pd.concat([resumen_gld, resumen_sca], ignore_index=True)
tabla_final = tabla_final.drop(columns=['antecedente'])
tabla_final.to_csv(OUT_DIR / 'resumen_reglas_prism.csv', index=False, encoding='utf-8-sig')

print('Archivos exportados en', OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob('*.json')):
    print(' -', p.name)
print(' -', 'resumen_reglas_prism.csv')


Archivos exportados en C:\Users\josuf\Desktop\Proyecto_Semaforizacion_Inteligente_mejorado\01_Reglas_PRISM
 - reglas_prism_green_light_duration.json
 - reglas_prism_signal_cycle_adjustment.json
 - umbrales_discretizacion.json
 - resumen_reglas_prism.csv



## Paso 11 — Conclusiones de la extracción de reglas

* PRISM identificó, de forma totalmente basada en datos, que la combinación
 **`vehicle_count = Alto` Y `average_speed = Alta` Y `traffic_density = Alta`** produce
 `green_light_duration = Larga` con **confianza perfecta (1.00)** — coincide con la regla
 destacada en la Sección VII.5 del documento (vías arteriales/autopistas de varios carriles
 con flujo veloz pero voluminoso).
* Se completó el recubrimiento secuencial total para las 3 clases de `green_light_duration` y
 las 3 clases de la variable derivada `signal_cycle_adjustment`, cerrando los dos puntos que
 el documento original dejaba como trabajo futuro (Sección IX).
* Las reglas exportadas en esta carpeta son el **único insumo de reglas** que utilizará el
 sistema difuso MIMO de la Carpeta 2 — no se reintroducen allí las reglas cualitativas de la
 Sección VI del documento, tal como fue solicitado.
